# 🚗 Nexus ALPR - OCR Model Training Pipeline
**Enterprise Edition - Data Science & ML Engineering Workflow**

This notebook handles the end-to-end training pipeline for the Optical Character Recognition (OCR) model.
It is specifically configured to detect and read Egyptian Arabic license plate characters using YOLOv8.

### 📋 Pipeline Architecture:
1. **Environment Setup:** Installing necessary dependencies (Ultralytics, Roboflow).
2. **Data Ingestion:** Fetching the annotated dataset securely via Roboflow API.
3. **Model Training:** YOLOv8s custom fine-tuning with specific augmentations (e.g., Disabled horizontal flipping to preserve Arabic character integrity).
4. **Production Export:** Exporting the trained weights to ONNX format for high-speed, CPU-optimized inference in the FastAPI backend.

In [ ]:
# ==========================================
# 📦 1. ENVIRONMENT SETUP & DEPENDENCIES
# ==========================================
print("Installing Ultralytics and Roboflow...")
!pip install -q ultralytics roboflow
print("Dependencies installed successfully! 🚀")

## 📥 2. Dataset Acquisition
Connecting to Roboflow to download the annotated Egyptian Car Plates dataset directly into the Colab environment.

In [ ]:
# ==========================================
# 📥 2. DATASET ACQUISITION (ROBOFLOW)
# ==========================================
from roboflow import Roboflow

# Initialize Roboflow with API Key
rf = Roboflow(api_key="**************")

# Fetch the specific project and version
project = rf.workspace("smartparkingsystem-jwynr").project("egyptian-car-plates-8jl5r")
version = project.version(2)

# Download the dataset in YOLOv8 format
dataset = version.download("yolov8")
print(f"Dataset downloaded to: {dataset.location}")

## 🧠 3. Model Training (YOLOv8s)
Training the OCR model.
> **⚠️ ML Engineering Note:** Horizontal and vertical flips (`fliplr=0.0`, `flipud=0.0`) are strictly disabled. This is a critical OCR practice to prevent the model from learning reversed or upside-down Arabic characters, which would corrupt inference.

In [ ]:
# ==========================================
# 🧠 3. CORE ML TRAINING PIPELINE
# ==========================================
from ultralytics import YOLO

# Initialize the YOLOv8 small model architecture
model_ocr = YOLO('yolov8s.pt')

# Execute the training loop with OCR-specific augmentations
results_ocr = model_ocr.train(
    data='/content/egyptian-car-plates-2/data.yaml',
    epochs=100,
    imgsz=320,               # Smaller image size optimized for cropped plates
    batch=16,

    # ⚠️ CRITICAL OCR AUGMENTATIONS ⚠️
    fliplr=0.0,              # Disabled: Arabic text loses meaning if flipped horizontally
    flipud=0.0,              # Disabled: Text shouldn't be upside down
    mosaic=0.5,              # Moderate mosaic to handle varied plate layouts

    # Lighting augmentations (useful for different times of day/camera glares)
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.6,

    patience=20              # Early stopping to prevent overfitting
)

## 📦 4. Model Export (ONNX)
Exporting the trained PyTorch model to ONNX format. ONNX (Open Neural Network Exchange) provides significantly faster inference times on CPUs and is heavily optimized for our `FastAPI` production environment.

In [ ]:
# ==========================================
# 📦 4. PRODUCTION EXPORT (ONNX)
# ==========================================

# Note: In a fresh run, you typically load from 'runs/detect/train/weights/best.pt'
ocr_model = YOLO('/content/runs/detect/train/weights/best.pt')

print("Exporting OCR Model for Production...")

# Exporting to ONNX format.
# dynamic=False ensures fixed input sizes (320px) for maximum inference speed.
ocr_model.export(format='onnx', imgsz=320, dynamic=False)

print("Export Complete! 🟢 The ONNX model is ready for the Nexus ALPR Backend.")